In [ ]:
import excalibuhr
import matplotlib.pyplot as plt
import numpy as np
from exocrires import plotMatrix
from excalibuhr.data import DETECTOR
from astropy.io import fits
from matplotlib.colors import SymLogNorm
import os 

In [ ]:
#Initialize pipeline
from excalibuhr import pipeline

workpath = './2023-01-01/chi_tau'
night ='2023-01-01'

ppl = pipeline.CriresPipeline(
                        workpath, night=night, obs_mode='nod',
                        num_processes=4, clean_start=False
                        )


In [ ]:
ppl.night='2023-01-01'
ppl.extract_header()
ppl.cal_dark()
ppl.cal_flat_raw()
ppl.cal_flat_trace()
ppl.cal_slit_curve()
ppl.cal_flat_norm()
ppl.obs_nodding()
ppl.obs_nodding_combine() #optional

In [ ]:
ppl.obs_extract(aper_comp=None)

In [ ]:
ppl.refine_wlen_solution(debug=True)

In [ ]:
ppl.run_molecfit()

In [ ]:
ppl.run_recipes(run_molecfit=True)

In [ ]:
extr2d_A = DETECTOR(filename='./2023-01-01/chi_tau/2023-01-01/out/combined/Extr2D_PRIMARY__COMBINED_chiTau_K2166_NOD_A.npz')
extr2d_B = DETECTOR(filename='./2023-01-01/chi_tau/2023-01-01/out/combined/Extr2D_PRIMARY__COMBINED_chiTau_K2166_NOD_B.npz')

data = fits.open('./2023-01-01/cal/WLEN_K2166_V_DH_Tau_A+B_center.fits')
hdu_wave = data[1]

flux2d_A = extr2d_A.flux
flux2d_B = extr2d_B.flux

variance2d_A = extr2d_A.var
variance2d_B = extr2d_B.var

psf2d_A = extr2d_A.psf
psf2d_B = extr2d_B.psf

Nodding = ['A', 'B']
i=0
aper_comp=int(np.ceil(60-(2.3/0.059)))
for flux2d in [flux2d_A, flux2d_B]:

    for order in range(0,7):

        fig, axes = plt.subplots(3, 3, figsize=(30, 10))
        fig.suptitle('Nodding %s, Order  %s' %(Nodding[i], (23 + order)), fontsize=16)


        for detector in range(0,3):

            #plotMatrix.plotMatrix(flux2d_A[0][order], [0,2048], np.arange(0, 20, 1), 'Wavelength Axis', 'Spatial Axis', planet_posi=10, scale='log')
            wave=hdu_wave.data[detector, order]

            ax = axes[detector,0]
            im = ax.imshow(flux2d[detector][order], aspect='auto', norm=SymLogNorm(5, 500))

            # Set x-axis label and ticks
            x_ticks = np.arange(0, wave.size, step=300)
            ax.set_xticks(x_ticks)
            ax.set_xticklabels(wave[x_ticks].round(2), fontsize=8)

            if detector == 2:
                ax.set_xlabel('Wavelength (nm)')

            cbar = plt.colorbar(im, ax=ax)
            #cbar.set_ticks([-10, 0, 10])
            #cbar.set_ticklabels([-10,0,10])
            
            ax2 = axes[detector,1]
            ax2.plot(flux2d[detector][order][aper_comp-1])
            ax2.set_ylim(-10, 30)

            ax2.set_xticks(ax.get_xticks())
            ax2.set_xticklabels(ax.get_xticklabels())

            ax3 = axes[detector,2]
            spatial_curve=np.trapz(flux2d[detector][order][:,], x=wave, axis=1)
            ax3.plot(spatial_curve)

            ax3.vlines(x=60, ymin=0, ymax=np.nanmax(spatial_curve), linestyles='--', colors='grey')
            ax3.vlines(x=aper_comp-1, ymin=0, ymax=np.nanmax(spatial_curve), linestyles='--', colors='grey')


            ax3.set_yscale('symlog')
            ax3.set_ylim(np.nanmax([np.nanmin(spatial_curve),5]), np.nanmax(spatial_curve)*1.5)
            ax3.set_xlabel('Spatial axis')


        
        plt.show()
    i+=1
    #plotMatrix.plotMatrix(flux2d_A[0][order]+flux2d_B[0][order], [0,2048], np.arange(0, 40, 1), 'Wavelength Axis', 'Spatial Axis', planet_posi=None, scale='log')
    #plt.title('Nodding A + B, Order  %s'%(29-order))